# 03 — DBSCAN Clustering
**Project:** GB Compound Hazards Risk Prioritization  
**Course:** Artificial Intelligence in Human Water Systems  

---
## Objectives
- Apply DBSCAN separately to extreme precipitation and wind gust points
- Characterize each cluster: duration, spatial extent, intensity, density
- Visualize cluster footprints over Great Britain
- Save cluster databases for compound detection in notebook 04


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.cluster import DBSCAN
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load Preprocessed Data

In [ ]:
# Load point arrays
rain_points = np.load('../data/processed/rain_points.npy')
rain_values = np.load('../data/processed/rain_values.npy')
wind_points = np.load('../data/processed/wind_points.npy')
wind_values = np.load('../data/processed/wind_values.npy')

# Load parameters
with open('../data/processed/dbscan_params.json') as f:
    PARAMS = json.load(f)

with open('../data/processed/time_references.json') as f:
    TIME_REF = json.load(f)

t0_rain = pd.Timestamp(TIME_REF['rain'])
t0_wind = pd.Timestamp(TIME_REF['wind'])

print(f'Rain points: {rain_points.shape}')
print(f'Wind points: {wind_points.shape}')
print(f'Parameters: {PARAMS}')

## 3. Run DBSCAN

> **Note:** DBSCAN on millions of points can take several minutes. Progress is shown below.

In [ ]:
def run_dbscan(points, eps, min_samples, label=''):
    """
    Run DBSCAN on space-time point array.
    Returns cluster labels (-1 = noise).
    """
    print(f'Running DBSCAN on {label} ({len(points):,} points)...')
    print(f'  eps={eps}, min_samples={min_samples}')
    
    db = DBSCAN(
        eps=eps,
        min_samples=min_samples,
        algorithm='ball_tree',
        n_jobs=-1  # Use all CPU cores
    ).fit(points)
    
    labels = db.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)
    
    print(f'  Clusters found: {n_clusters:,}')
    print(f'  Noise points:   {n_noise:,} ({100*n_noise/len(points):.1f}%)')
    
    return labels, n_clusters


rain_labels, n_rain_clusters = run_dbscan(
    rain_points,
    eps=PARAMS['rain']['eps'],
    min_samples=PARAMS['rain']['min_samples'],
    label='Precipitation'
)

print()

wind_labels, n_wind_clusters = run_dbscan(
    wind_points,
    eps=PARAMS['wind']['eps'],
    min_samples=PARAMS['wind']['min_samples'],
    label='Wind Gust'
)

## 4. Build Cluster Databases

For each cluster we compute: start/end time, duration, spatial extent, peak intensity, mean intensity, density.

In [ ]:
def build_cluster_database(points, values, labels, t0, resolution=0.25, hazard_type='precipitation'):
    """
    Build a cluster database from DBSCAN output.
    
    Parameters:
    -----------
    points     : normalized space-time points (lat, lon, hours)
    values     : intensity values per point
    labels     : DBSCAN cluster labels
    t0         : reference timestamp for time reconstruction
    resolution : spatial grid resolution in degrees
    hazard_type: 'precipitation' or 'wind'
    
    Returns:
    --------
    pd.DataFrame with one row per cluster
    """
    unique_labels = sorted(set(labels))
    if -1 in unique_labels:
        unique_labels.remove(-1)  # Exclude noise
    
    records = []
    
    for cluster_id in tqdm(unique_labels, desc=f'Building {hazard_type} cluster DB'):
        mask = labels == cluster_id
        cluster_points = points[mask]
        cluster_values = values[mask]
        
        # Reconstruct real coordinates
        lats = cluster_points[:, 0] * resolution
        lons = cluster_points[:, 1] * resolution
        hours = cluster_points[:, 2]
        
        # Time attributes
        start_time = t0 + pd.Timedelta(hours=float(hours.min()))
        end_time   = t0 + pd.Timedelta(hours=float(hours.max()))
        duration_h = float(hours.max() - hours.min())
        
        # Spatial attributes
        n_cells = len(set(zip(lats.round(2), lons.round(2))))
        # Approximate area: each cell is ~0.25° lat x 0.25° lon
        # At ~54°N: 1° lat ≈ 111 km, 1° lon ≈ 65 km
        cell_area_km2 = (0.25 * 111) * (0.25 * 65)
        spatial_extent_km2 = n_cells * cell_area_km2
        
        # Intensity attributes
        peak_intensity = float(cluster_values.max())
        mean_intensity = float(cluster_values.mean())
        
        # Density (points per hour)
        density = len(cluster_points) / max(duration_h, 1)
        
        # Season
        month = start_time.month
        if month in [12, 1, 2]:   season = 'DJF'
        elif month in [3, 4, 5]:  season = 'MAM'
        elif month in [6, 7, 8]:  season = 'JJA'
        else:                      season = 'SON'
        
        records.append({
            'cluster_id': cluster_id,
            'hazard_type': hazard_type,
            'start_time': start_time,
            'end_time': end_time,
            'duration_h': duration_h,
            'n_points': int(mask.sum()),
            'n_cells': n_cells,
            'spatial_extent_km2': spatial_extent_km2,
            'peak_intensity': peak_intensity,
            'mean_intensity': mean_intensity,
            'density': density,
            'lat_min': float(lats.min()),
            'lat_max': float(lats.max()),
            'lon_min': float(lons.min()),
            'lon_max': float(lons.max()),
            'lat_centroid': float(lats.mean()),
            'lon_centroid': float(lons.mean()),
            'year': start_time.year,
            'season': season
        })
    
    return pd.DataFrame(records)


rain_clusters_db = build_cluster_database(
    rain_points, rain_values, rain_labels, t0_rain, hazard_type='precipitation'
)

wind_clusters_db = build_cluster_database(
    wind_points, wind_values, wind_labels, t0_wind, hazard_type='wind'
)

print(f'\nPrecipitation clusters: {len(rain_clusters_db):,}')
print(f'Wind clusters:          {len(wind_clusters_db):,}')

## 5. Cluster Summary Statistics

In [ ]:
print('=== Precipitation Clusters Summary ===')
print(rain_clusters_db[['duration_h', 'spatial_extent_km2', 'peak_intensity', 'mean_intensity', 'density']].describe().round(3))

print('\n=== Wind Clusters Summary ===')
print(wind_clusters_db[['duration_h', 'spatial_extent_km2', 'peak_intensity', 'mean_intensity', 'density']].describe().round(3))

## 6. Visualize Cluster Footprints

In [ ]:
def plot_cluster_footprints(cluster_db, points, labels, title, color='steelblue', sample_n=20):
    """
    Plot spatial footprints of the largest clusters.
    """
    # Select top N largest clusters by spatial extent
    top_clusters = cluster_db.nlargest(sample_n, 'spatial_extent_km2')['cluster_id'].values
    resolution = 0.25
    
    fig, ax = plt.subplots(figsize=(8, 10))
    
    for cid in top_clusters:
        mask = labels == cid
        lats = points[mask, 0] * resolution
        lons = points[mask, 1] * resolution
        ax.scatter(lons, lats, s=1, alpha=0.3, color=color)
    
    ax.set_title(f'{title}\n(Top {sample_n} clusters by spatial extent)', fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xlim(-8, 4)
    ax.set_ylim(46, 61)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()


plot_cluster_footprints(
    rain_clusters_db, rain_points, rain_labels,
    'Precipitation Cluster Footprints', color='steelblue'
)
plt.savefig('../outputs/figures/03_rain_cluster_footprints.png', dpi=150, bbox_inches='tight')
plt.show()

plot_cluster_footprints(
    wind_clusters_db, wind_points, wind_labels,
    'Wind Gust Cluster Footprints', color='darkorange'
)
plt.savefig('../outputs/figures/03_wind_cluster_footprints.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Clusters per Year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rain_per_year = rain_clusters_db.groupby('year').size()
wind_per_year = wind_clusters_db.groupby('year').size()

axes[0].bar(rain_per_year.index, rain_per_year.values, color='steelblue', edgecolor='white')
axes[0].set_title('Precipitation Clusters per Year', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of clusters')

axes[1].bar(wind_per_year.index, wind_per_year.values, color='darkorange', edgecolor='white')
axes[1].set_title('Wind Gust Clusters per Year', fontsize=13)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of clusters')

plt.tight_layout()
plt.savefig('../outputs/figures/03_clusters_per_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Cluster Databases

In [ ]:
rain_clusters_db.to_parquet('../data/processed/rain_clusters_db.parquet', index=False)
wind_clusters_db.to_parquet('../data/processed/wind_clusters_db.parquet', index=False)

# Also save labels for compound detection
np.save('../data/processed/rain_labels.npy', rain_labels)
np.save('../data/processed/wind_labels.npy', wind_labels)

print('Saved:')
print(f'  rain_clusters_db.parquet — {len(rain_clusters_db):,} clusters')
print(f'  wind_clusters_db.parquet — {len(wind_clusters_db):,} clusters')
print('  rain_labels.npy')
print('  wind_labels.npy')